In [1]:
# Load env variables and create client
from dotenv import load_dotenv

load_dotenv()

# Create an API client
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-0"

In [3]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

# make a request
def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params = {
        "model" : model,
        "max_tokens" : 1000,
        "messages" : messages,
        "temperature" : temperature,
    }

    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return message.content[0].text

In [2]:
import json

def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [ ]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [4]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [5]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    # TODO = Grading
    score = 10

    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [6]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    return results

In [7]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [8]:
print(json.dumps(results, indent=2))

[
  {
    "output": "I'll write a Python function to validate AWS S3 bucket names according to the AWS naming conventions.\n\n```python\nimport re\n\ndef is_valid_s3_bucket_name(bucket_name):\n    \"\"\"\n    Validates if a bucket name follows AWS S3 naming conventions.\n    \n    AWS S3 bucket naming rules:\n    - Must be between 3 and 63 characters long\n    - Can consist only of lowercase letters, numbers, dots (.), and hyphens (-)\n    - Must begin and end with a letter or number\n    - Must not contain two adjacent periods\n    - Must not be formatted as an IP address (e.g., 192.168.5.4)\n    - Must not start with 'xn--' (reserved for internationalized domain names)\n    - Must not end with '-s3alias' (reserved for access point aliases)\n    \n    Args:\n        bucket_name (str): The bucket name to validate\n        \n    Returns:\n        bool: True if valid, False otherwise\n    \"\"\"\n    \n    # Check if input is a string\n    if not isinstance(bucket_name, str):\n        re